
# 09: Backpropagation Intuition — How Networks Learn

## The "Blame Game"

Backpropagation is how neural networks learn. The intuition is simple:

1. **Make a prediction** (forward pass)
2. **Calculate the error** (how wrong were we?)
3. **Assign blame** (which weights caused the error?)
4. **Update weights** (fix the mistakes)

### The Web Dev Analogy

It's like debugging a pipeline:
- You see wrong output (error)
- You trace back through the code
- You find which function caused the bug
- You fix that function

Backpropagation automates this "blame assignment"!

---

### Why learn this manually when Lesson 11 automates it?

**Lesson 11 (autograd)** will handle backprop automatically — you'll never write `dW2 = ...` in real PyTorch code. So why learn it here?

Two reasons:

1. **Debugging**: When your model doesn't learn, you'll want to know: *Are gradients flowing? Are they vanishing? Is a layer receiving zero gradient?* You can't debug what you don't understand.

2. **Interviews and depth**: Every ML engineer is expected to know "how backprop works." This lesson gives you that.

> **If this feels abstract**: Focus on the *intuition* (blame assignment, chain rule direction), not the math symbols. Skim the code, read the comments, and know that Lesson 11 will make it click in a completely different way.


## What You'll Learn
- [ ] Explain the chain rule and how gradients flow backward through layers
- [ ] Implement one step of backpropagation manually
- [ ] Verify computed gradients numerically using finite differences

## Connection to Previous Lessons

| What you learned | How it connects here |
|-----------------|---------------------|
| **Lesson 3**: Gradient descent (compute gradient → update weights) | Same idea, but now gradients must flow through *multiple layers* |
| **Lesson 8**: Multi-layer networks | How do we train them? Backpropagation chains the gradient through every layer |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print("Ready to understand backpropagation! 🔙")

## 1. The Chain Rule: The Key to Backpropagation

Backpropagation is just the **chain rule** from calculus, applied systematically.

If $y = f(g(x))$, then:
$$\frac{dy}{dx} = \frac{dy}{dg} \cdot \frac{dg}{dx}$$

**Don't worry if calculus isn't your thing!** The intuition is what matters:

> "How much does changing X affect Y?" = (How much does X affect the middle) × (How much does the middle affect Y)

In [ ]:
# Simple example: f(x) = (2x + 1)²
# This is f(g(x)) where g(x) = 2x + 1 and f(g) = g²

def g(x):
    return 2*x + 1

def f(g_val):
    return g_val ** 2

def full_function(x):
    return f(g(x))  # (2x + 1)²

# At x = 3:
x = 3
g_val = g(x)  # 2*3 + 1 = 7
y = f(g_val)  # 7² = 49

# Chain rule:
# dy/dx = (dy/dg) * (dg/dx)
#       = (2g) * (2)
#       = 2*7 * 2 = 28

dy_dg = 2 * g_val  # Derivative of g² is 2g
dg_dx = 2          # Derivative of 2x+1 is 2
dy_dx = dy_dg * dg_dx

print(f"At x = {x}:")
print(f"  g(x) = 2x + 1 = {g_val}")
print(f"  f(g) = g² = {y}")
print(f"\nChain Rule:")
print(f"  dy/dg = 2g = {dy_dg}")
print(f"  dg/dx = 2")
print(f"  dy/dx = {dy_dg} × {dg_dx} = {dy_dx}")

# Verify numerically
epsilon = 0.0001
numerical_gradient = (full_function(x + epsilon) - full_function(x)) / epsilon
print(f"\nNumerical verification: {numerical_gradient:.4f} ≈ {dy_dx}")

## 2. Forward Pass: Computing Predictions

Let's trace through a simple network step by step:

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

# Simple network: 2 inputs → 2 hidden → 1 output
# Let's use fixed weights for illustration

# Weights
W1 = np.array([[0.5, -0.5], [0.3, 0.8]])  # 2x2
b1 = np.array([[0.1, -0.2]])               # 1x2
W2 = np.array([[0.7], [-0.4]])             # 2x1
b2 = np.array([[0.2]])                      # 1x1

# Input and target
X = np.array([[1.0, 0.5]])  # One sample
y_true = np.array([[1.0]])  # Target

print("=" * 50)
print("FORWARD PASS")
print("=" * 50)

# Layer 1: Linear
z1 = np.dot(X, W1) + b1
print(f"\n1. Hidden layer linear: z1 = X @ W1 + b1")
print(f"   z1 = {z1}")

# Layer 1: Activation
a1 = sigmoid(z1)
print(f"\n2. Hidden layer activation: a1 = sigmoid(z1)")
print(f"   a1 = {a1}")

# Layer 2: Linear
z2 = np.dot(a1, W2) + b2
print(f"\n3. Output layer linear: z2 = a1 @ W2 + b2")
print(f"   z2 = {z2}")

# Layer 2: Activation (output)
a2 = sigmoid(z2)
print(f"\n4. Output layer activation: a2 = sigmoid(z2)")
print(f"   a2 = {a2} (our prediction!)")

# Loss
loss = -np.mean(y_true * np.log(a2) + (1 - y_true) * np.log(1 - a2))
print(f"\n5. Loss: {loss:.4f}")
print(f"   (Target was {y_true[0,0]}, we predicted {a2[0,0]:.4f})")

## 3. Backward Pass: Computing Gradients

Now we go backwards, computing how much each weight contributed to the error:

In [ ]:
print("=" * 50)
print("BACKWARD PASS")
print("=" * 50)

# Start from the output, work backwards

# Step 1: Gradient of loss with respect to a2 (output)
# For binary cross-entropy: dL/da2 = -y/a2 + (1-y)/(1-a2)
# But with sigmoid output, it simplifies to: dL/dz2 = a2 - y
dz2 = a2 - y_true
print(f"\n1. Output error: dz2 = a2 - y_true")
print(f"   dz2 = {dz2}")
print(f"   (This is how 'wrong' our output was)")

# Step 2: Gradient for W2 and b2
# dL/dW2 = a1.T @ dz2 (how much each W2 weight contributed)
# dL/db2 = sum(dz2)
dW2 = np.dot(a1.T, dz2)
db2 = np.sum(dz2, axis=0, keepdims=True)
print(f"\n2. Output layer gradients:")
print(f"   dW2 = a1.T @ dz2 = {dW2.flatten()}")
print(f"   db2 = {db2}")

# Step 3: Propagate error back to hidden layer
# dz1 = (dz2 @ W2.T) * sigmoid_derivative(z1)
da1 = np.dot(dz2, W2.T)  # Error flowing back through W2
dz1 = da1 * sigmoid_derivative(z1)  # Through the activation
print(f"\n3. Error backpropagated to hidden layer:")
print(f"   da1 = dz2 @ W2.T = {da1}")
print(f"   dz1 = da1 * sigmoid'(z1) = {dz1}")

# Step 4: Gradient for W1 and b1
dW1 = np.dot(X.T, dz1)
db1 = np.sum(dz1, axis=0, keepdims=True)
print(f"\n4. Hidden layer gradients:")
print(f"   dW1 = X.T @ dz1 = \n{dW1}")
print(f"   db1 = {db1}")

## 4. Visualizing the Blame Flow

In [ ]:
# Create a visual representation
fig, ax = plt.subplots(figsize=(14, 8))

# Draw network structure
layers = [
    [(0.15, 0.7), (0.15, 0.3)],  # Input
    [(0.45, 0.7), (0.45, 0.3)],  # Hidden
    [(0.75, 0.5)],               # Output
]

layer_names = ['Input', 'Hidden', 'Output']
node_values = [
    [f'x₁={X[0,0]:.1f}', f'x₂={X[0,1]:.1f}'],
    [f'a₁={a1[0,0]:.3f}', f'a₂={a1[0,1]:.3f}'],
    [f'ŷ={a2[0,0]:.3f}']
]
gradient_values = [
    ['', ''],
    [f'δ={dz1[0,0]:.4f}', f'δ={dz1[0,1]:.4f}'],
    [f'δ={dz2[0,0]:.4f}']
]

# Draw nodes
for i, (layer, name, vals, grads) in enumerate(zip(layers, layer_names, node_values, gradient_values)):
    for j, ((x, y), val, grad) in enumerate(zip(layer, vals, grads)):
        # Node
        circle = plt.Circle((x, y), 0.06, color='lightblue', ec='black', linewidth=2)
        ax.add_patch(circle)
        
        # Value (forward pass - blue)
        ax.text(x, y+0.02, val, ha='center', va='center', fontsize=9, color='blue')
        
        # Gradient (backward pass - red)
        if grad:
            ax.text(x, y-0.02, grad, ha='center', va='center', fontsize=8, color='red')
    
    # Layer name
    ax.text(layer[0][0], 0.95, name, ha='center', fontsize=12, fontweight='bold')

# Draw connections
for i in range(len(layers)-1):
    for node1 in layers[i]:
        for node2 in layers[i+1]:
            ax.annotate('', xy=(node2[0]-0.06, node2[1]), xytext=(node1[0]+0.06, node1[1]),
                       arrowprops=dict(arrowstyle='->', color='gray', alpha=0.5))

# Add arrows showing flow direction
ax.annotate('Forward →', xy=(0.45, 0.1), fontsize=12, color='blue',
           ha='center', fontweight='bold')
ax.annotate('← Backward', xy=(0.45, 0.05), fontsize=12, color='red',
           ha='center', fontweight='bold')

# Target and loss
ax.text(0.9, 0.5, f'Target: {y_true[0,0]}\nLoss: {loss:.4f}', fontsize=10,
       bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='black'))

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('Backpropagation: Forward Pass (blue) and Backward Pass (red)', fontsize=14)

plt.tight_layout()
plt.show()

print("Blue values: computed during forward pass")
print("Red values (δ): gradients computed during backward pass")

## 5. The Weight Update Step

In [ ]:
learning_rate = 0.5

print("Weight Updates:")
print("=" * 50)

# Update rule: W_new = W_old - learning_rate * gradient

print(f"\nW2 (before): {W2.flatten()}")
W2_new = W2 - learning_rate * dW2
print(f"W2 (after):  {W2_new.flatten()}")
print(f"Change:      {(-learning_rate * dW2).flatten()}")

print(f"\nb2 (before): {b2.flatten()}")
b2_new = b2 - learning_rate * db2
print(f"b2 (after):  {b2_new.flatten()}")

print(f"\nW1 (before):\n{W1}")
W1_new = W1 - learning_rate * dW1
print(f"W1 (after):\n{W1_new}")

print(f"\nb1 (before): {b1}")
b1_new = b1 - learning_rate * db1
print(f"b1 (after):  {b1_new}")

In [ ]:
# Verify the update improved our prediction!
print("\nVerifying improvement:")
print("=" * 50)

# Forward pass with new weights
z1_new = np.dot(X, W1_new) + b1_new
a1_new = sigmoid(z1_new)
z2_new = np.dot(a1_new, W2_new) + b2_new
a2_new = sigmoid(z2_new)

loss_new = -np.mean(y_true * np.log(a2_new) + (1 - y_true) * np.log(1 - a2_new))

print(f"Before update: prediction = {a2[0,0]:.4f}, loss = {loss:.4f}")
print(f"After update:  prediction = {a2_new[0,0]:.4f}, loss = {loss_new:.4f}")
print(f"\nTarget: {y_true[0,0]}")
print(f"\n✅ Prediction got closer to target!")
print(f"✅ Loss decreased by {loss - loss_new:.4f}!")

## 6. Complete Training Loop

In [ ]:
class SimpleNN:
    """Neural network with explicit backprop for learning."""
    
    def __init__(self):
        # Initialize weights
        np.random.seed(42)
        self.W1 = np.random.randn(2, 4) * 0.5
        self.b1 = np.zeros((1, 4))
        self.W2 = np.random.randn(4, 1) * 0.5
        self.b2 = np.zeros((1, 1))
        self.history = []
    
    def forward(self, X):
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = sigmoid(self.z1)
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = sigmoid(self.z2)
        return self.a2
    
    def backward(self, X, y, lr=0.5):
        m = X.shape[0]
        
        # Backward pass
        dz2 = self.a2 - y
        dW2 = (1/m) * np.dot(self.a1.T, dz2)
        db2 = (1/m) * np.sum(dz2, axis=0, keepdims=True)
        
        da1 = np.dot(dz2, self.W2.T)
        dz1 = da1 * sigmoid_derivative(self.z1)
        dW1 = (1/m) * np.dot(X.T, dz1)
        db1 = (1/m) * np.sum(dz1, axis=0, keepdims=True)
        
        # Update
        self.W2 -= lr * dW2
        self.b2 -= lr * db2
        self.W1 -= lr * dW1
        self.b1 -= lr * db1
    
    def train(self, X, y, epochs=1000, lr=1.0):
        for epoch in range(epochs):
            # Forward
            pred = self.forward(X)
            
            # Loss
            loss = -np.mean(y * np.log(pred + 1e-15) + (1-y) * np.log(1-pred + 1e-15))
            self.history.append(loss)
            
            # Backward
            self.backward(X, y, lr)
            
            if epoch % 200 == 0:
                acc = np.mean((pred > 0.5) == y)
                print(f"Epoch {epoch:4d}: Loss = {loss:.4f}, Acc = {acc:.2%}")

# Train on XOR
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([[0], [1], [1], [0]])

print("Training on XOR with explicit backpropagation:\n")
nn = SimpleNN()
nn.train(X_xor, y_xor, epochs=2000, lr=2.0)

# Results
print("\nFinal predictions:")
preds = nn.forward(X_xor)
for x, y, p in zip(X_xor, y_xor, preds):
    print(f"  {x} → {p[0]:.3f} (target: {y[0]})")

In [ ]:
# Plot training
plt.figure(figsize=(10, 4))
plt.plot(nn.history)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Backpropagation Learning XOR')
plt.grid(True, alpha=0.3)
plt.show()

## 7. Key Intuitions

### The Blame Analogy

Imagine a factory assembly line:
1. **Forward pass**: Materials flow through stations, creating a product
2. **Quality check**: The product has defects (error)
3. **Backward pass**: Trace back - which station caused each defect?
4. **Fix**: Adjust each station proportionally to its blame

### Why Gradients?

The gradient tells us:
- **Direction**: Which way to change the weight (+ or -)
- **Magnitude**: How much this weight contributed to error

In [ ]:
# Visualize gradient magnitudes
print("Gradient magnitudes (how much each weight 'contributed' to error):")
print("=" * 60)

# Using our original example
print(f"\nOutput layer (W2): {np.abs(dW2).flatten()}")
print(f"Hidden layer (W1): \n{np.abs(dW1)}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

im1 = axes[0].imshow(np.abs(dW1), cmap='Reds')
axes[0].set_title('|dW1| - Hidden Layer Gradients')
axes[0].set_xlabel('Hidden Neuron')
axes[0].set_ylabel('Input')
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(np.abs(dW2), cmap='Reds')
axes[1].set_title('|dW2| - Output Layer Gradients')
axes[1].set_xlabel('Output')
axes[1].set_ylabel('Hidden Neuron')
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

print("\n💡 Larger gradient = more blame = bigger update!")

## ⚠️ What Can Go Wrong: Vanishing Gradients

In deep networks with sigmoid activation, gradients **shrink exponentially**
as they flow backward. Sigmoid's maximum derivative is only 0.25 — so each
layer multiplies the gradient by at most 0.25. After 5 layers: 0.25⁵ ≈ 0.001!

In [ ]:
# --- Vanishing gradients in a deep sigmoid network ---
# Simulate gradient flow through layers with sigmoid activation
# sigmoid'(z) has max value of 0.25 (at z=0)

n_layers = 8
gradient = 1.0  # Start with gradient of 1 at the output
gradient_magnitudes = [gradient]

print("Gradient magnitudes flowing backward through 8 sigmoid layers:")
print("=" * 60)
print(f"  Output layer: gradient = {gradient:.6f}")

for layer in range(n_layers):
    # Each sigmoid layer multiplies gradient by at most sigmoid'(z)
    # Average case: sigmoid'(z) ≈ 0.2 for typical activations
    sigmoid_deriv = 0.20  # Typical value during training
    gradient *= sigmoid_deriv
    gradient_magnitudes.append(gradient)
    print(f"  Layer {n_layers - layer} → Layer {n_layers - layer - 1}: "
          f"gradient = {gradient:.8f}  (×{sigmoid_deriv})")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Linear scale
layers = list(range(n_layers, -1, -1))
axes[0].bar(range(len(gradient_magnitudes)), gradient_magnitudes, color='steelblue')
axes[0].set_xlabel('Layer (output → input)')
axes[0].set_ylabel('Gradient Magnitude')
axes[0].set_title('Vanishing Gradients (Linear Scale)')

# Log scale
axes[1].bar(range(len(gradient_magnitudes)), gradient_magnitudes, color='coral')
axes[1].set_xlabel('Layer (output → input)')
axes[1].set_ylabel('Gradient Magnitude (log)')
axes[1].set_title('Vanishing Gradients (Log Scale)')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

print(f"\n📉 Gradient shrinks from 1.0 to {gradient_magnitudes[-1]:.8f}")
print(f"   That's {1/gradient_magnitudes[-1]:.0f}x smaller!")
print(f"\n🔑 This is why ReLU replaced sigmoid in deep networks:")
print(f"   ReLU derivative = 1 (for positive inputs) → no shrinking!")
print(f"   Also why residual connections (Lesson 25) help: gradients can skip layers.")

## 📝 Check Your Understanding

1. What is the chain rule and why does backprop need it?
2. What does a gradient represent?
3. Why do we multiply by the learning rate?
4. What happens if a gradient is zero?
5. Why do we need activation functions to have derivatives?

In [ ]:
# --- Exercise 1: Numerical Gradient Check ---
# Verify the derivative of f(x) = x² at x=3 using finite differences.
# Analytical answer: f'(x) = 2x = 6
# Numerical: (f(x+h) - f(x-h)) / (2h)

def f(x):
    return x ** 2

x = 3.0
h = 0.0001

# YOUR CODE HERE:
numerical_grad = None  # Compute (f(x+h) - f(x-h)) / (2*h)
analytical_grad = None  # Compute 2*x

# --- Check ---
assert numerical_grad is not None and analytical_grad is not None, "Compute both!"
assert abs(numerical_grad - 6.0) < 0.001, f"Numerical gradient should be ~6.0, got {numerical_grad:.6f}"
assert abs(analytical_grad - 6.0) < 0.001, f"Analytical gradient should be 6.0, got {analytical_grad}"
assert abs(numerical_grad - analytical_grad) < 0.001, "Numerical and analytical should match!"
print(f"Exercise 1 passed! ✓  (Numerical: {numerical_grad:.6f}, Analytical: {analytical_grad:.1f})")

# --- Quick Check: Chain Rule ---
# If f(x) = sigmoid(3x + 2), what is df/dx?
# We need the chain rule: df/dx = sigmoid'(z) × d(3x+2)/dx
# where z = 3x + 2 and sigmoid'(z) = sigmoid(z)(1-sigmoid(z))
# So df/dx = sigmoid(z)(1-sigmoid(z)) × 3
#
# The chain rule lets us compute gradients through...
# a) Only one layer at a time
# b) Composed functions by multiplying local derivatives
# c) Only linear functions
# d) Functions without using calculus

your_answer = None  # Put 'a', 'b', 'c', or 'd'

# --- Check ---
assert your_answer is not None, "Pick an answer!"
assert your_answer == 'b', "Chain rule: d(f∘g)/dx = f'(g(x)) × g'(x) — multiply the local derivatives!"
print("Exercise 2 passed! ✓")

print("\n🎉 All exercises passed!")

## 🎯 Summary

Backpropagation:
1. **Forward pass**: Compute predictions layer by layer
2. **Compute loss**: How wrong were we?
3. **Backward pass**: Chain rule to get gradients
4. **Update weights**: Move opposite to gradients

Key insight: **Gradients = blame assignment**
- Larger gradient → more blame → bigger update
- Sign of gradient → direction to move

**Next up**: PyTorch - where autograd does backprop for us! →